# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates how to load and analyze the FAIR^2 dataset, which contains ordered logistic regression results for predictors of adoption of indigenous and modern knowledge in rangeland management in Northern Kenya, using the `mlcroissant` library.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Let's review the available **record sets**, their fields, and their unique `@id` strings, which are critical for querying and referencing elements in this dataset. 

_All references below use `@id` fields per Croissant best practices._

In [ ]:
# Retrieve all record sets with their @id, name, and field details
record_sets = list(dataset.record_sets())
print(f"Found {len(record_sets)} record sets.")
record_set_ids = []

for rs in record_sets:
    print(f"\nRecord set name: {rs.name}")
    print(f"  @id: {rs.id}")
    record_set_ids.append(rs.id)
    
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}, type: {field.data_type}, column: {field.column.id if getattr(field, 'column', None) else 'N/A'})")

## 3. Data Extraction
We'll now load data from each record set into a Pandas DataFrame. All record and field references use their `@id`.

Below, each loaded DataFrame is keyed by the record set `@id`.

In [ ]:
# Load each record set as DataFrame using its @id
dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Preview columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print("  No records found!")
# For demonstration, select the first available record set for further analysis
if dataframes:
    selected_record_set_id = next(iter(dataframes))
    df = dataframes[selected_record_set_id]
    print(f"\nSelected record set for EDA: {selected_record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())
else:
    selected_record_set_id = None
    print("No dataframes available for analysis.")

## 4. Exploratory Data Analysis (EDA)
Let's perform basic EDA: filter based on a numeric field, normalize it, and group by a key attribute.

If you want to use a specific field, refer to its `@id` (the column name will correspond to field `@id`).

In [ ]:
# --- EDA parameters ---

if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]
    # Guess a numeric field from the DataFrame columns
    numeric_candidates = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]  # Use the first numeric
        print(f"Numeric field selected for analysis: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() == df[numeric_field_id].mean() else 0  # Use mean or 0 if NaN
        threshold = threshold if threshold else 10  # If mean is 0, fallback to 10

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with '{numeric_field_id}' > {threshold:.2f} (total: {len(filtered_df)}):")
        print(filtered_df.head())

        # Normalize the selected numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to select a group_by field (use any non-numeric column)
        group_candidates = [col for col in df.columns if col != numeric_field_id and df[col].dtype == object]
        group_field_id = group_candidates[0] if group_candidates else None

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped data by '{group_field_id}':")
            print(grouped_df.head())
        else:
            print("\nNo suitable categorical field available for grouping.")
    else:
        print("No numeric fields available in the selected record set.")
else:
    print("No record set selected for EDA.")

## 5. Visualization
Visualize the data distribution and relationships. We'll plot the main numeric field and, if possible, show summary by a category.

_Modify the plot as needed for your record set columns._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id is not None and 'numeric_field_id' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('No visualizations available - check data and field selection.')

## 6. Conclusion
In this notebook, we loaded the FAIR^2 ordered logistic regression results dataset defined in Croissant using the `mlcroissant` library. We:
- Explored available record sets and fields, referencing all data by their `@id`s.
- Loaded records as Pandas DataFrames for each record set.
- Conducted basic exploratory and statistical analyses on the numeric fields.
- Visualized field distributions and relationships.

This workflow demonstrates best practices for FAIR data exploration and use of Croissant-compliant datasets in scientific data science.

_For further analysis, consult the Croissant schema for all available fields and `@id`s to ensure correct and consistent usage._